# 3. `warning` (gige, temperature)

In [1]:
from common import *

shape, log = show(3)

log shape 96: gige | temperature | warning
49,157 logs (11.9%) from 8 nodes, 2003-12-26 12:36:12 to 2006-04-30 11:23:24
log 208661 at 2005-03-20 02:26:10: 124215 gige3 gige temperature 1111285570 1 warning


### This log warns about the temperature of a gige device (by its name a gigabit ethernet switch, there are 8 of them), which is a concern.
But it doesnt say the temperature - therefore I don't know the severity yet.

What this device logged in the day before

In [2]:
day_before = timeline(log.node, log.ts - timedelta(days=1), log.ts)
display(day_before)
assert set(day_before.template) == {"warning", "normal"}

,first_ts,last_ts,count,component,state,template
0,2005-03-19 03:46:08,2005-03-19 03:46:08,1,gige,temperature,normal
1,2005-03-19 03:56:08,2005-03-19 03:56:08,1,gige,temperature,warning
2,2005-03-19 07:21:08,2005-03-19 07:21:08,1,gige,temperature,normal
3,2005-03-19 07:46:10,2005-03-19 07:46:10,1,gige,temperature,warning
4,2005-03-19 09:16:08,2005-03-19 09:16:08,1,gige,temperature,normal
5,2005-03-19 09:26:09,2005-03-19 09:26:09,1,gige,temperature,warning
6,2005-03-19 10:11:09,2005-03-19 10:11:09,1,gige,temperature,normal
7,2005-03-19 10:16:09,2005-03-19 10:16:09,1,gige,temperature,warning
8,2005-03-19 13:21:10,2005-03-19 13:21:10,1,gige,temperature,normal
9,2005-03-19 13:31:10,2005-03-19 13:31:10,1,gige,temperature,warning


#### Ok, it alternates between 'warning' and 'normal' states.

Minutes spent at each level in that day (a line is written when the level changes)

In [3]:
at_level = minutes_at_level(day_before)
display(at_level)
share_in_warning = at_level.loc["warning", "total"] / at_level.total.sum()
print(f"in 'warning' {share_in_warning:.0%} of the day")
assert share_in_warning > 0.7

,times,median,longest,total
template,,,,
normal,18,7.5,25.0,155.1
warning,17,65.0,205.0,1204.9


in 'warning' 89% of the day


Level changes of every gige device in the 30 days before

In [4]:
levels = gige_levels(log.ts - 30 * day, log.ts)
display(levels)
assert levels.loc[log.node, "critical"] <= 1

,normal,warning,critical
gige1,192,192,0
gige2,563,564,0
gige3,378,380,1
gige4,31,499,489
gige5,686,687,0
gige6,171,253,86
gige7,280,590,379
gige8,0,1,0


# Verdict: looks like low severity, since this isn't something new. We should raise a warning to the user to check this gige out.
'warning' is where this device lives most of the time, and so do most other giges.
In addition, this device almost never reaches critical. So we are good. Warn the user once to check the gige, lowest severity.